# PayOrFix - Is your Attrition a Paycheck Problem or a Structural Flaw?

### This Project uses the IBM HR Dataset to identify department-level root cause for attrition and its possible solution.

> ##### Hypothesis to be tested:
> ##### 1. Attrition in Sales Department is mostly driven by structural variables.
> ##### 2. Attrition in R&D is driven by satisfaction and involvement.
> ##### 3. The pay gap between stayers and leavers will be small, indicating that it is structure that drives attrition instead of paychecks.

### A. Data Cleaning

In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv("../data/Raw/raw_data.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

In [4]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [5]:
df.columns.tolist()

['Age',
 'Attrition',
 'BusinessTravel',
 'DailyRate',
 'Department',
 'DistanceFromHome',
 'Education',
 'EducationField',
 'EmployeeCount',
 'EmployeeNumber',
 'EnvironmentSatisfaction',
 'Gender',
 'HourlyRate',
 'JobInvolvement',
 'JobLevel',
 'JobRole',
 'JobSatisfaction',
 'MaritalStatus',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'Over18',
 'OverTime',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StandardHours',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

In [6]:
# Remove unwanted columns
drop_cols = [
    'Age',
    'DailyRate',
    'DistanceFromHome',
    'Education',
    'EducationField',
    'EmployeeCount',
    'EmployeeNumber',
    'Gender',
    'HourlyRate',
    'JobRole',
    'MaritalStatus',
    'MonthlyRate',
    'NumCompaniesWorked',
    'Over18',
    'PerformanceRating',
    'StandardHours',
    'TotalWorkingYears',
    'TrainingTimesLastYear',
    'YearsAtCompany',
    'YearsInCurrentRole',
    'YearsWithCurrManager'
]
dfn = df.drop(columns=drop_cols)

> *These columns do not affect the factors involved in answering the business question.*

In [7]:
dfn.head()

,Attrition,BusinessTravel,Department,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,OverTime,PercentSalaryHike,RelationshipSatisfaction,StockOptionLevel,WorkLifeBalance,YearsSinceLastPromotion
0,Yes,Travel_Rarely,Sales,2,3,2,4,5993,Yes,11,1,0,1,0
1,No,Travel_Frequently,Research & Development,3,2,2,2,5130,No,23,4,1,3,1
2,Yes,Travel_Rarely,Research & Development,4,2,1,3,2090,Yes,15,2,0,3,0
3,No,Travel_Frequently,Research & Development,4,3,1,3,2909,Yes,11,3,0,3,3
4,No,Travel_Rarely,Research & Development,1,3,1,2,3468,No,12,4,1,3,2


In [8]:
# Check for null values
dfn.isnull()

,Attrition,BusinessTravel,Department,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,OverTime,PercentSalaryHike,RelationshipSatisfaction,StockOptionLevel,WorkLifeBalance,YearsSinceLastPromotion
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1466,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1467,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1468,False,False,False,False,False,False,False,False,False,False,False,False,False,False


> *No Null values. Confirmed.*

In [9]:
dfn.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Attrition                 1470 non-null   str  
 1   BusinessTravel            1470 non-null   str  
 2   Department                1470 non-null   str  
 3   EnvironmentSatisfaction   1470 non-null   int64
 4   JobInvolvement            1470 non-null   int64
 5   JobLevel                  1470 non-null   int64
 6   JobSatisfaction           1470 non-null   int64
 7   MonthlyIncome             1470 non-null   int64
 8   OverTime                  1470 non-null   str  
 9   PercentSalaryHike         1470 non-null   int64
 10  RelationshipSatisfaction  1470 non-null   int64
 11  StockOptionLevel          1470 non-null   int64
 12  WorkLifeBalance           1470 non-null   int64
 13  YearsSinceLastPromotion   1470 non-null   int64
dtypes: int64(10), str(4)
memory usage: 160.9 KB


In [10]:
dfn.describe()

,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,PercentSalaryHike,RelationshipSatisfaction,StockOptionLevel,WorkLifeBalance,YearsSinceLastPromotion
count,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000
mean,2.721769,2.729932,2.063946,2.728571,6502.931293,15.209524,2.712245,0.793878,2.761224,2.187755
std,1.093082,0.711561,1.106940,1.102846,4707.956783,3.659938,1.081209,0.852077,0.706476,3.222430
min,1.000000,1.000000,1.000000,1.000000,1009.000000,11.000000,1.000000,0.000000,1.000000,0.000000
25%,2.000000,2.000000,1.000000,2.000000,2911.000000,12.000000,2.000000,0.000000,2.000000,0.000000
50%,3.000000,3.000000,2.000000,3.000000,4919.000000,14.000000,3.000000,1.000000,3.000000,1.000000
75%,4.000000,3.000000,3.000000,4.000000,8379.000000,18.000000,4.000000,1.000000,3.000000,3.000000
max,4.000000,4.000000,5.000000,4.000000,19999.000000,25.000000,4.000000,3.000000,4.000000,15.000000


In [11]:
# Number of Departments
dfn['Department'].value_counts()

Department
Research & Development    961
Sales                     446
Human Resources            63
Name: count, dtype: int64

> *Three departments in total. Noted.*

In [12]:
# Check for Duplicate rows
dfn.duplicated().sum()

np.int64(0)

> *No Duplicate values. Confirmed.*

### B. Data Encoding and Normalisation

In [13]:
# Encode Attrition column
dfn['Attrition_bin'] = (df['Attrition'] == 'Yes').astype(int)

# Encode OverTime for numerical analysis
dfn['OverTime_bin'] = (df['OverTime'] == 'Yes').astype(int)

# Encode BusinessTravel as ordinal
travel_map = {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
dfn['BusinessTravel_ord'] = df['BusinessTravel'].map(travel_map)

In [14]:
# Define Structural and Compensation Variables
dfs = dfn[['OverTime_bin', 'WorkLifeBalance', 'JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction', 'JobInvolvement', 'BusinessTravel_ord']].copy()

dfc = dfn[['MonthlyIncome', 'PercentSalaryHike', 'StockOptionLevel', 'JobLevel']].copy()

In [15]:
# Normalize the Data
num_cols = dfn.select_dtypes(include='number').columns.tolist()
exclude = ['Attrition_bin', 'OverTime_bin', 'BusinessTravel_ord']
cols_to_scale = [col for col in num_cols if col not in exclude]

# Create normalised copy
df_norm = dfn.copy()
scaler = MinMaxScaler()
df_norm[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# Remove encoded Columns from Normalized Data
df_norm = df_norm.drop(columns=['Attrition', 'BusinessTravel', 'OverTime'])

In [16]:
# Export the cleaned data to csv
dfn.to_csv('../data/Processed/PayOrFixCleanedData.csv', index=False)
print('Cleaned File Exported!')
df_norm.to_csv('../data/Processed/PayOrFixNormalizedData.csv', index=False)
print('Normalized Data Exported!')

Cleaned File Exported!
Normalized Data Exported!


In [17]:
dfn.info()
df_norm.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Attrition                 1470 non-null   str  
 1   BusinessTravel            1470 non-null   str  
 2   Department                1470 non-null   str  
 3   EnvironmentSatisfaction   1470 non-null   int64
 4   JobInvolvement            1470 non-null   int64
 5   JobLevel                  1470 non-null   int64
 6   JobSatisfaction           1470 non-null   int64
 7   MonthlyIncome             1470 non-null   int64
 8   OverTime                  1470 non-null   str  
 9   PercentSalaryHike         1470 non-null   int64
 10  RelationshipSatisfaction  1470 non-null   int64
 11  StockOptionLevel          1470 non-null   int64
 12  WorkLifeBalance           1470 non-null   int64
 13  YearsSinceLastPromotion   1470 non-null   int64
 14  Attrition_bin             1470 non-null   int64
 15